# ChessHint 2.0.0 — Stockfish 19 no Colab
1. Escolha um ambiente **CPU** e execute a célula **Instalar**. Ela contém o servidor desta versão: não baixa código do repositório original.
2. Execute **Iniciar**, copie a **URL PARA O PAINEL** e a **CHAVE PARA O PAINEL**.
3. Abra uma partida no Roblox e execute o conteúdo de `chess.lua` do pacote ChessHint 2.0.0.
4. Escolha seu lado e quem joga agora, cole URL e chave, e clique **Iniciar / resincronizar**.

O notebook instala binários oficiais com SHA256 verificado e executa testes antes de iniciar. Os modos Preciso e Atacante funcionam aqui. URL/chave são específicas desta sessão; mantenha a célula Iniciar ativa. Interrompê-la encerra o servidor e o túnel. O Colab pode encerrar o runtime; se isso acontecer, execute as células novamente. Comece preferencialmente antes da primeira jogada para preservar todo o histórico. Para partida em andamento, a FEN completa opcional recupera roque e en passant conhecidos.


In [ ]:
# Python / Colab installation / CPU runtime / ChessHint 2.0.0.
import os, sys, subprocess, pathlib, urllib.request, hashlib
BASE = pathlib.Path('/content/chesshint-v2')
BASE.mkdir(exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'chess==1.11.2'], check=True)
target = BASE / 'termux/server.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('#!/usr/bin/env python3\n"""Python / server.py / Python 3.10+ / authenticated Stockfish + legal sessions."""\nimport argparse\nimport hmac\nimport json\nimport math\nimport secrets\nimport threading\nimport logging\nimport signal\nfrom positions import PositionStore, PositionError\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nfrom pathlib import Path\nimport chess\nimport chess.engine\n\nROOT = Path(__file__).resolve().parent\nVERSION = "2.0.0"\n\ndef attack_value(board, move):\n    side = board.turn\n    value = 0.0\n    captured = board.piece_at(move.to_square)\n    moving = board.piece_at(move.from_square)\n    if captured:\n        value += 1.0\n    if moving:\n        rank = chess.square_rank(move.to_square)\n        value += (rank if side == chess.WHITE else 7 - rank) * 0.08\n    after = board.copy()\n    after.push(move)\n    if after.is_check():\n        value += 1.5\n    king = after.king(not side)\n    if king is not None:\n        ring = set(after.attacks(king)) | {king}\n        value += sum(len(after.attackers(side, sq)) for sq in ring) * 0.2\n    # Material offered is only a preference inside the evaluation-loss budget.\n    if moving and after.is_attacked_by(not side, move.to_square):\n        value += {chess.PAWN:0.1,chess.KNIGHT:0.3,chess.BISHOP:0.3,\n                  chess.ROOK:0.5,chess.QUEEN:0.8}.get(moving.piece_type, 0)\n    return value\n\ndef select_line(board, lines, style, aggression):\n    lines = [line for line in lines if line.get("score") is not None and line.get("pv")\n             and line["pv"][0] in board.legal_moves]\n    if not lines:\n        raise RuntimeError("Motor não retornou linha válida")\n    ranked = sorted(lines, key=lambda line: line["score"].pov(board.turn).score(mate_score=100000), reverse=True)\n    best = ranked[0]\n    best_score = best["score"].pov(board.turn)\n    # Preserve forced mate decisions instead of treating mate as centipawns.\n    if style == "precise" or aggression == 0 or best_score.is_mate():\n        return best, 0\n    top = best_score.score()\n    budget = round(aggression * 0.5)  # 100 => at most 50 centipawns.\n    eligible = [line for line in ranked\n                if not line["score"].pov(board.turn).is_mate()\n                and top - line["score"].pov(board.turn).score() <= budget]\n    chosen = max(eligible, key=lambda line:\n                 attack_value(board, line["pv"][0]) * (aggression / 100)\n                 - (top - line["score"].pov(board.turn).score()) / 100)\n    return chosen, top - chosen["score"].pov(board.turn).score()\n\n\nclass Service:\n    def __init__(self, engine, key, force_precise=False, adaptive_time=False):\n        self.engine, self.key = engine, key\n        self.lock = threading.Lock()\n        self.force_precise = force_precise\n        self.adaptive_time = adaptive_time\n        self.positions = PositionStore()\n\n    def analyse(self, payload):\n        if not isinstance(payload, dict):\n            raise ValueError("JSON deve ser um objeto")\n        session_result = None\n        if "session" in payload:\n            with self.positions.lock:\n                session = self.positions._session(payload["session"])\n                revision = payload.get("revision")\n                if type(revision) is not int or revision != session.revision:\n                    raise PositionError("STALE", "Revisão da análise desatualizada")\n                board = session.board.copy(stack=True)\n                session_result = self.positions._result(payload["session"], session)\n            fen = board.fen(en_passant="fen")\n        else:\n            fen = payload.get("fen")\n        if not isinstance(fen, str) or len(fen) > 256 or len(fen.split()) != 6:\n            raise ValueError("Envie FEN completa com seis campos")\n        if session_result is None:\n            board = chess.Board(fen)\n        if not board.is_valid():\n            raise ValueError(f"Posição inválida; status={int(board.status())}")\n        if isinstance(payload.get("time"), bool) or isinstance(payload.get("aggression"), bool):\n            raise ValueError("Tempo e agressividade devem ser números")\n        seconds = float(payload.get("time", 1))\n        if not math.isfinite(seconds) or not 0.1 <= seconds <= 10:\n            raise ValueError("Tempo deve estar entre 0.1 e 10 segundos")\n        style = payload.get("style", "precise")\n        if style not in ("precise", "attacking"):\n            raise ValueError("Estilo deve ser precise ou attacking")\n        aggression = float(payload.get("aggression", 0))\n        if not math.isfinite(aggression) or not 0 <= aggression <= 100:\n            raise ValueError("Agressividade deve estar entre 0 e 100")\n        if self.force_precise:\n            style, aggression = "precise", 0\n        requested_seconds = seconds\n        legal = list(board.legal_moves)\n        # Conservative heuristic: tactical positions retain the full budget.\n        tactical = board.is_check() or any(board.is_capture(m) or m.promotion or board.gives_check(m) for m in legal)\n        if self.adaptive_time and not tactical:\n            seconds = max(0.1, requested_seconds * 0.75)\n        if self.adaptive_time and len(legal) == 1:\n            seconds = min(seconds, 0.1)\n        outcome = board.outcome()\n        if outcome:\n            return {"game_over": True, "result": outcome.result(),\n                    "reason": outcome.termination.name, "fen": fen,\n                    **(session_result or {})}\n        if not self.lock.acquire(blocking=False):\n            return None\n        try:\n            count = min(4, board.legal_moves.count()) if style == "attacking" and aggression > 0 else 1\n            lines = self.engine.analyse(board, chess.engine.Limit(time=seconds), multipv=count)\n            info, loss = select_line(board, lines, style, aggression)\n            pv = info.get("pv", [])\n            if not pv or pv[0] not in board.legal_moves:\n                raise RuntimeError("Motor não retornou jogada legal")\n            score = info.get("score")\n            white = score.white() if score is not None else None\n            cp = white.score() if white is not None else None\n            return {"move": pv[0].uci(), "depth": info.get("depth"),\n                    "eval": cp / 100 if cp is not None else None,\n                    "mate": white.mate() if white is not None else None,\n                    "pv": [m.uci() for m in pv], "nodes": info.get("nodes"),\n                    "time": info.get("time"), "engine": self.engine.id.get("name"),\n                    "fen": fen, "game_over": False, "style": style,\n                    "aggression": aggression, "evaluation_loss_cp": loss,\n                    "personality_supported": not self.force_precise, "multipv": count,\n                    "requested_time": requested_seconds, "search_budget": seconds,\n                    "adaptive_time": self.adaptive_time, "version": VERSION,\n                    "claimable_draw": board.can_claim_draw(), **(session_result or {})}\n        finally:\n            self.lock.release()\n\nclass Handler(BaseHTTPRequestHandler):\n    protocol_version = "HTTP/1.1"\n\n    def setup(self):\n        super().setup()\n        self.connection.settimeout(15)\n\n    def reply(self, status, data):\n        body = json.dumps(data, ensure_ascii=False).encode()\n        self.send_response(status)\n        self.send_header("Content-Type", "application/json; charset=utf-8")\n        self.send_header("Content-Length", str(len(body)))\n        self.send_header("Cache-Control", "no-store")\n        if status == 429:\n            self.send_header("Retry-After", "1")\n        self.send_header("Connection", "close")\n        self.end_headers()\n        self.close_connection = True\n        try:\n            self.wfile.write(body)\n        except (BrokenPipeError, ConnectionResetError):\n            pass\n\n    def authorized(self):\n        expected = ("Bearer " + self.server.service.key).encode()\n        actual = self.headers.get("Authorization", "").encode()\n        if not hmac.compare_digest(actual, expected):\n            self.reply(401, {"error": "UNAUTHORIZED", "text": "Chave incorreta"})\n            return False\n        return True\n\n    def do_GET(self):\n        if not self.authorized():\n            return\n        if self.path != "/health":\n            self.reply(404, {"error": "NOT_FOUND"})\n            return\n        self.reply(200, {"ok": True, "engine": self.server.service.engine.id,\n                         "version": VERSION, "sessions": True,\n                         "personality_supported": not self.server.service.force_precise})\n\n    def do_POST(self):\n        if not self.authorized():\n            return\n        if self.path not in ("/analyse", "/session", "/sync", "/close"):\n            self.reply(404, {"error": "NOT_FOUND"})\n            return\n        try:\n            length = int(self.headers.get("Content-Length", "0"))\n            if not 0 < length <= 4096:\n                self.reply(413, {"error": "INVALID_SIZE"})\n                return\n            self.connection.settimeout(15)\n            payload = json.loads(self.rfile.read(length))\n            if not isinstance(payload, dict):\n                raise ValueError("JSON deve ser objeto")\n            service = self.server.service\n            if self.path == "/session":\n                result = service.positions.create(payload.get("placement"), payload.get("turn"), payload.get("initial_fen"))\n            elif self.path == "/sync":\n                result = service.positions.sync(payload.get("session"), payload.get("placement"), payload.get("revision"))\n            elif self.path == "/close":\n                service.positions.delete(payload.get("session"))\n                result = {"ok": True}\n            else:\n                result = service.analyse(payload)\n            self.reply(429, {"error": "BUSY", "retry_after": 1}) if result is None else self.reply(200, result)\n        except PositionError as exc:\n            self.reply(409, {"error": exc.code, "text": str(exc)})\n        except (ValueError, TypeError, UnicodeError, OverflowError) as exc:\n            self.reply(400, {"error": "INVALID_REQUEST", "text": str(exc)})\n        except TimeoutError:\n            self.reply(408, {"error": "TIMEOUT"})\n        except Exception as exc:\n            print(f"Erro no motor: {type(exc).__name__}: {exc}", flush=True)\n            self.reply(503, {"error": "ENGINE_ERROR", "text": "Reinicie o servidor"})\n\n    def log_message(self, fmt, *args):\n        logging.info(fmt, *args)\n\n\nclass BoundedServer(ThreadingHTTPServer):\n    daemon_threads = True\n\n    def __init__(self, address, handler, capacity=16):\n        self.slots = threading.BoundedSemaphore(capacity)\n        super().__init__(address, handler)\n\n    def process_request(self, request, address):\n        if not self.slots.acquire(blocking=False):\n            self.shutdown_request(request)\n            return\n        try:\n            super().process_request(request, address)\n        except BaseException:\n            self.slots.release()\n            raise\n\n    def process_request_thread(self, request, address):\n        try:\n            super().process_request_thread(request, address)\n        finally:\n            self.slots.release()\n\n\nclass EngineManager:\n    """Replace a terminated UCI process once, under Service\'s analysis lock."""\n    def __init__(self, path, threads, hash_mb):\n        self.path = path\n        self.options = {"Threads": threads, "Hash": hash_mb,\n                        "Skill Level": 20, "UCI_LimitStrength": False}\n        self.engine = None\n        self._start()\n\n    @property\n    def id(self):\n        return self.engine.id\n\n    def _start(self):\n        if self.engine is not None:\n            self.engine.close()\n        self.engine = chess.engine.SimpleEngine.popen_uci(self.path, timeout=20)\n        try:\n            self.engine.configure(self.options)\n        except BaseException:\n            self.engine.close()\n            raise\n\n    def analyse(self, *args, **kwargs):\n        for attempt in range(2):\n            try:\n                return self.engine.analyse(*args, **kwargs)\n            except (chess.engine.EngineTerminatedError, chess.engine.EngineError,\n                    TimeoutError, BrokenPipeError):\n                if attempt:\n                    raise\n                logging.warning("Reiniciando Stockfish após falha UCI")\n                self._start()\n\n    def close(self):\n        self.engine.close()\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--engine", default=str(ROOT / "bin" / "stockfish"))\n    parser.add_argument("--threads", type=int, default=2)\n    parser.add_argument("--hash", type=int, default=128)\n    parser.add_argument("--port", type=int, default=8080)\n    parser.add_argument("--force-precise", action="store_true")\n    parser.add_argument("--adaptive-time", action="store_true")\n    args = parser.parse_args()\n    if not 1 <= args.threads <= 16 or not 16 <= args.hash <= 65536:\n        parser.error("threads: 1–16; hash: 16–65536 MB")\n    key_path = ROOT / ".api-key"\n    if not key_path.exists():\n        key_path.write_text(secrets.token_urlsafe(32) + "\\n")\n    key_path.chmod(0o600)\n    key = key_path.read_text().strip()\n    if not key:\n        raise SystemExit("Arquivo .api-key vazio")\n    engine = EngineManager(args.engine, args.threads, args.hash)\n    server = None\n    def shutdown_signal(signum, frame):\n        raise KeyboardInterrupt\n    signal.signal(signal.SIGTERM, shutdown_signal)\n    signal.signal(signal.SIGINT, shutdown_signal)\n    try:\n        server = BoundedServer(("127.0.0.1", args.port), Handler)\n        server.service = Service(engine, key, args.force_precise, args.adaptive_time)\n        print("Motor:", engine.id.get("name"), flush=True)\n        print("ChessHint", VERSION, flush=True)\n        print("Chave para colar no painel:", key, flush=True)\n        print(f"Servidor: http://127.0.0.1:{args.port}", flush=True)\n        server.serve_forever()\n    except KeyboardInterrupt:\n        pass\n    finally:\n        if server:\n            server.server_close()\n        engine.close()\n\nif __name__ == "__main__":\n    main()\n\n\n', encoding='utf-8')
target = BASE / 'termux/positions.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text("# Python / positions.py / Python 3.10+ / legal chess snapshot reconciliation.\nimport secrets\nimport threading\nimport time\nfrom dataclasses import dataclass\n\nimport chess\n\n\nclass PositionError(ValueError):\n    def __init__(self, code, text):\n        super().__init__(text)\n        self.code = code\n\n\n@dataclass\nclass Session:\n    board: chess.Board\n    history_complete: bool\n    touched: float\n    revision: int = 0\n    last_move: str | None = None\n\n\nclass PositionStore:\n    def __init__(self, max_sessions=32, ttl=7200):\n        self.sessions = {}\n        self.lock = threading.RLock()\n        self.max_sessions = max_sessions\n        self.ttl = ttl\n\n    @staticmethod\n    def placement(value):\n        if not isinstance(value, str) or len(value) > 100 or ' ' in value:\n            raise PositionError('INVALID_POSITION', 'Envie somente as oito fileiras da posição')\n        try:\n            board = chess.Board(value + ' w - - 0 1')\n        except ValueError as exc:\n            raise PositionError('INVALID_POSITION', str(exc)) from exc\n        if board.board_fen() != value:\n            raise PositionError('INVALID_POSITION', 'Posição deve usar FEN canônica')\n        return value\n\n    def _purge(self):\n        now = time.monotonic()\n        for key in list(self.sessions):\n            if now - self.sessions[key].touched > self.ttl:\n                del self.sessions[key]\n\n    def _session(self, key):\n        self._purge()\n        if not isinstance(key, str) or key not in self.sessions:\n            raise PositionError('SESSION_EXPIRED', 'Sessão encerrada; clique em Resincronizar')\n        session = self.sessions[key]\n        session.touched = time.monotonic()\n        return session\n\n    @staticmethod\n    def _result(key, session):\n        return {'session': key, 'fen': session.board.fen(en_passant='fen'),\n                'revision': session.revision, 'last_move': session.last_move,\n                'history_complete': session.history_complete}\n\n    def create(self, placement, turn, initial_fen=None):\n        placement = self.placement(placement)\n        if turn not in ('w', 'b'):\n            raise PositionError('INVALID_POSITION', 'Vez deve ser w ou b')\n        complete = False\n        if initial_fen:\n            if not isinstance(initial_fen, str) or len(initial_fen) > 256 or len(initial_fen.split()) != 6:\n                raise PositionError('INVALID_POSITION', 'FEN inicial deve ter seis campos')\n            try:\n                board = chess.Board(initial_fen)\n            except ValueError as exc:\n                raise PositionError('INVALID_POSITION', str(exc)) from exc\n            if board.board_fen() != placement:\n                raise PositionError('DESYNC', 'A FEN informada não corresponde às peças observadas')\n            complete = board.fen() == chess.STARTING_FEN\n        elif placement == chess.Board().board_fen():\n            board = chess.Board()\n            complete = True\n        else:\n            board = chess.Board(placement + ' ' + turn + ' - - 0 1')\n        if not board.is_valid():\n            raise PositionError('INVALID_POSITION', f'Posição inválida: status={int(board.status())}')\n        with self.lock:\n            self._purge()\n            if len(self.sessions) >= self.max_sessions:\n                raise PositionError('SESSION_LIMIT', 'Limite de sessões; reinicie o servidor ou aguarde expiração')\n            key = secrets.token_urlsafe(24)\n            session = Session(board, complete, time.monotonic())\n            self.sessions[key] = session\n            return self._result(key, session)\n\n    def sync(self, key, placement, revision=None):\n        placement = self.placement(placement)\n        with self.lock:\n            session = self._session(key)\n            if revision is not None and (type(revision) is not int or revision != session.revision):\n                raise PositionError('STALE', 'Revisão mudou; confira o tabuleiro e resincronize')\n            if session.board.board_fen() == placement:\n                return self._result(key, session)\n            if len(session.board.move_stack) >= 2048:\n                raise PositionError('HISTORY_LIMIT', 'Histórico excedeu o limite; resincronize')\n            matches = []\n            for move in session.board.legal_moves:\n                after = session.board.copy(stack=False)\n                after.push(move)\n                if after.board_fen() == placement:\n                    matches.append(move)\n            # A slow castle may expose a stable rook-only snapshot first.\n            # Correct only the immediately preceding corner-rook displacement,\n            # and only when the full new snapshot is a legal castle from before it.\n            if not matches and session.board.move_stack:\n                before = session.board.copy(stack=True)\n                previous = before.pop()\n                piece = before.piece_at(previous.from_square)\n                rank = 0 if before.turn == chess.WHITE else 7\n                rook_pairs = {(chess.square(7, rank), chess.square(5, rank)),\n                              (chess.square(0, rank), chess.square(3, rank))}\n                if piece and piece.piece_type == chess.ROOK and (previous.from_square, previous.to_square) in rook_pairs:\n                    for castle in list(before.legal_moves):\n                        if not before.is_castling(castle):\n                            continue\n                        after = before.copy(stack=False)\n                        after.push(castle)\n                        if after.board_fen() == placement:\n                            before.push(castle)\n                            session.board = before\n                            session.last_move = castle.uci()\n                            session.revision += 1\n                            return self._result(key, session)\n            if len(matches) != 1:\n                raise PositionError('DESYNC', 'Tabuleiro não corresponde a uma jogada legal; espere a animação ou resincronize')\n            session.board.push(matches[0])\n            session.last_move = matches[0].uci()\n            session.revision += 1\n            return self._result(key, session)\n\n    def get(self, key):\n        with self.lock:\n            return self._session(key).board.copy(stack=True)\n\n    def delete(self, key):\n        with self.lock:\n            if isinstance(key, str):\n                self.sessions.pop(key, None)\n", encoding='utf-8')
target = BASE / 'termux/install_engine.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text("# Python / install_engine.py / Python 3.10+ / official Stockfish 19 installer.\nimport hashlib\nimport os\nimport platform\nimport shutil\nimport tarfile\nimport tempfile\nimport urllib.request\nimport zipfile\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parent\nRELEASE = 'sf_19'\nASSETS = {\n    'android-arm64': ('stockfish-android-arm64-universal.tar.gz', 'ebb24051aa4a222b4daaf049b882ecf1163d370c128fe02316602643f4d5e426'),\n    'android-armv7': ('stockfish-android-armv7-neon.tar.gz', '47c34963f1cdf4a6af34c1b5294f7a4e2679b3eb52698c324855af2c6486024b'),\n    'linux-arm64': ('stockfish-linux-arm64-universal.tar.gz', 'fe26cfd1d9db4c8af3d21e24d9ff34cacb31c1f940085a7583da11796f2bac01'),\n    'linux-x64': ('stockfish-linux-x86-64-universal.tar.gz', '9defc0d4e55d49c65a6d042f3e571a39fcea499ade6dbe741b53b8c65e03611f'),\n    'macos': ('stockfish-macos-universal.tar.gz', 'a1f0e3bcc5a6927a11fe6fc8e54a779754645f3c2bae2cf13420fd1957adaa77'),\n    'windows-x64': ('stockfish-windows-x86-64-universal.zip', '3c8bf1f9ea66a09350a40df4f632288285ac206d99f33ab5842c408fc30b48a7'),\n    'windows-arm64': ('stockfish-windows-arm64-universal.zip', '8372ad3f0d7276deb2c70f801f541ec7db463219fc6d9c7592864e542aa4f401'),\n}\n\n\ndef asset_key():\n    machine = platform.machine().lower()\n    arch = 'arm64' if machine in ('aarch64', 'arm64') else 'x64' if machine in ('x86_64', 'amd64') else 'armv7'\n    if 'com.termux' in os.environ.get('PREFIX', '') or hasattr(os, 'getandroidapilevel'):\n        return 'android-' + arch\n    system = platform.system()\n    if system == 'Darwin':\n        return 'macos'\n    if system == 'Windows':\n        return 'windows-' + arch\n    if system == 'Linux' and arch != 'armv7':\n        return 'linux-' + arch\n    raise RuntimeError(f'Plataforma não suportada: {system}/{machine}')\n\n\ndef download(url, target):\n    request = urllib.request.Request(url, headers={'User-Agent': 'ChessHint/2.0'})\n    with urllib.request.urlopen(request, timeout=60) as source, target.open('wb') as dest:\n        shutil.copyfileobj(source, dest)\n\n\ndef install():\n    name, expected = ASSETS[asset_key()]\n    target = ROOT / 'bin' / ('stockfish.exe' if name.endswith('.zip') else 'stockfish')\n    marker = target.with_suffix('.release')\n    if target.is_file() and marker.is_file() and marker.read_text().strip() == expected:\n        print('Stockfish 19 já instalado:', target)\n        return target\n    target.parent.mkdir(exist_ok=True)\n    print('Baixando Stockfish 19 oficial; primeira instalação pode demorar...', flush=True)\n    with tempfile.TemporaryDirectory(dir=ROOT) as folder:\n        archive = Path(folder) / name\n        download(f'https://github.com/official-stockfish/Stockfish/releases/download/{RELEASE}/{name}', archive)\n        with archive.open('rb') as source:\n            digest = hashlib.file_digest(source, 'sha256').hexdigest() if hasattr(hashlib, 'file_digest') else hashlib.sha256(source.read()).hexdigest()\n        if digest != expected:\n            raise RuntimeError('SHA256 do Stockfish não corresponde à versão oficial')\n        temporary = Path(folder) / target.name\n        if name.endswith('.zip'):\n            with zipfile.ZipFile(archive) as package:\n                binaries = [m for m in package.infolist() if m.filename.endswith('.exe') and Path(m.filename).name.startswith('stockfish')]\n                if len(binaries) != 1:\n                    raise RuntimeError('Arquivo oficial sem binário único')\n                with package.open(binaries[0]) as source, temporary.open('wb') as dest:\n                    shutil.copyfileobj(source, dest)\n        else:\n            with tarfile.open(archive, 'r:gz') as package:\n                binaries = [m for m in package.getmembers() if m.isfile() and\n                            Path(m.name).name.startswith('stockfish-') and '.' not in Path(m.name).name]\n                if len(binaries) != 1:\n                    raise RuntimeError('Arquivo oficial sem binário único')\n                with package.extractfile(binaries[0]) as source, temporary.open('wb') as dest:\n                    shutil.copyfileobj(source, dest)\n        temporary.chmod(0o755)\n        os.replace(temporary, target)\n        marker.write_text(expected + '\\n')\n    print('Instalado:', target, flush=True)\n    return target\n\n\nif __name__ == '__main__':\n    install()\n", encoding='utf-8')
target = BASE / 'termux/check_engine.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('# Python / check_engine.py / Python 3.10+ / real Stockfish smoke tests.\nimport argparse\nimport chess\nfrom server import EngineManager, Service\n\n\ndef check(path):\n    engine = EngineManager(path, 1, 32)\n    service = Service(engine, \'test\')\n    count = 0\n    try:\n        for fen, style in [(chess.STARTING_FEN, \'precise\'),\n                           (chess.STARTING_FEN, \'attacking\'),\n                           (\'7k/5Q2/6K1/8/8/8/8/8 w - - 0 1\', \'precise\')]:\n            result = service.analyse({\'fen\': fen, \'time\': 0.1, \'style\': style, \'aggression\': 100})\n            board = chess.Board(fen)\n            move = chess.Move.from_uci(result[\'move\'])\n            assert move in board.legal_moves, result\n            assert 0 <= result[\'evaluation_loss_cp\'] <= 50, result\n            if fen.startswith(\'7k/\'):\n                board.push(move)\n                assert board.is_checkmate(), result\n            print(\'OK:\', style, result[\'move\'], \'profundidade\', result[\'depth\'])\n            count += 1\n        session = service.positions.create(chess.Board().board_fen(), \'w\')\n        board = chess.Board(); board.push_uci(\'e2e4\')\n        session = service.positions.sync(session[\'session\'], board.board_fen(), 0)\n        result = service.analyse({**session, \'time\': 0.1})\n        assert chess.Move.from_uci(result[\'move\']) in board.legal_moves\n        assert result[\'revision\'] == 1\n        count += 1\n        engine.engine.close()\n        result = service.analyse({\'fen\': chess.STARTING_FEN, \'time\': 0.1})\n        assert chess.Move.from_uci(result[\'move\']) in chess.Board().legal_moves\n        count += 1\n        print(f\'{count} verificações com {engine.id.get("name")} passaram.\')\n    finally:\n        engine.close()\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--engine\', required=True)\n    check(parser.parse_args().engine)\n', encoding='utf-8')
target = BASE / 'termux/requirements.txt'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('chess==1.11.2\n', encoding='utf-8')
target = BASE / 'tests/test_positions.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""Regression tests for legal, atomic visual-board reconciliation."""\nimport sys\nimport unittest\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \'termux\'))\nimport chess\nfrom positions import PositionStore, PositionError\n\n\nclass PositionTests(unittest.TestCase):\n    def setUp(self):\n        self.store = PositionStore(max_sessions=32, ttl=7200)\n\n    def create(self, fen):\n        board = chess.Board(fen)\n        return self.store.create(board.board_fen(), \'w\' if board.turn else \'b\', initial_fen=fen)\n\n    def advance(self, state, uci):\n        board = self.store.get(state[\'session\'])\n        board.push_uci(uci)\n        result = self.store.sync(state[\'session\'], board.board_fen(), revision=state[\'revision\'])\n        self.assertEqual(result[\'last_move\'], uci)\n        self.assertEqual(result[\'revision\'], state[\'revision\'] + 1)\n        self.assertEqual(result[\'fen\'], board.fen(en_passant=\'fen\'))\n        return result\n\n    def error(self, code, action):\n        with self.assertRaises(PositionError) as caught:\n            action()\n        self.assertEqual(caught.exception.code, code)\n\n    def test_starting_position_has_rights_and_complete_history(self):\n        result = self.store.create(chess.Board().board_fen(), \'w\')\n        self.assertEqual(result[\'fen\'], chess.STARTING_FEN)\n        self.assertTrue(result[\'history_complete\'])\n\n    def test_midgame_defaults_without_inferred_rights(self):\n        board = chess.Board(\'r3k2r/8/8/8/8/8/8/R3K2R w KQkq - 0 1\')\n        result = self.store.create(board.board_fen(), \'w\')\n        actual = self.store.get(result[\'session\'])\n        self.assertEqual(actual.castling_rights, 0)\n        self.assertIsNone(actual.ep_square)\n        self.assertFalse(result[\'history_complete\'])\n\n    def test_four_castlings_are_atomic(self):\n        for turn, move in [(\'w\', \'e1g1\'), (\'w\', \'e1c1\'), (\'b\', \'e8g8\'), (\'b\', \'e8c8\')]:\n            with self.subTest(move=move):\n                state = self.create(f\'r3k2r/8/8/8/8/8/8/R3K2R {turn} KQkq - 0 1\')\n                self.advance(state, move)\n\n    def test_rook_displacement_is_a_legal_rook_move(self):\n        state = self.create(\'r3k2r/8/8/8/8/8/8/R3K2R w KQkq - 0 1\')\n        self.advance(state, \'h1f1\')\n\n    def test_en_passant_from_sequence(self):\n        state = self.store.create(chess.Board().board_fen(), \'w\')\n        for move in [\'e2e4\', \'a7a6\', \'e4e5\', \'d7d5\', \'e5d6\']:\n            state = self.advance(state, move)\n        self.assertIsNone(self.store.get(state[\'session\']).piece_at(chess.D5))\n\n    def test_all_promotion_choices(self):\n        for piece in \'qrbn\':\n            with self.subTest(piece=piece):\n                state = self.create(\'7k/P7/8/8/8/8/8/7K w - - 0 1\')\n                self.advance(state, \'a7a8\' + piece)\n\n    def test_capture(self):\n        state = self.store.create(chess.Board().board_fen(), \'w\')\n        for move in [\'e2e4\', \'d7d5\', \'e4d5\']:\n            state = self.advance(state, move)\n\n    def test_idempotent_snapshot(self):\n        state = self.store.create(chess.Board().board_fen(), \'w\')\n        result = self.store.sync(state[\'session\'], chess.Board().board_fen(), revision=state[\'revision\'])\n        self.assertEqual(result[\'revision\'], state[\'revision\'])\n        self.assertEqual(result[\'fen\'], state[\'fen\'])\n        self.assertIsNone(result[\'last_move\'])\n\n    def test_illegal_displacement_does_not_mutate(self):\n        state = self.store.create(chess.Board().board_fen(), \'w\')\n        invalid = chess.Board()\n        invalid.set_piece_at(chess.E5, invalid.remove_piece_at(chess.E2))\n        self.error(\'DESYNC\', lambda: self.store.sync(state[\'session\'], invalid.board_fen()))\n        self.assertEqual(self.store.get(state[\'session\']).fen(en_passant=\'fen\'), state[\'fen\'])\n        valid = self.advance(state, \'e2e4\')\n        self.assertEqual(valid[\'revision\'], state[\'revision\'] + 1)\n\n    def test_pinned_piece_cannot_expose_king(self):\n        state = self.create(\'k3r3/8/8/8/8/8/4R3/4K3 w - - 0 1\')\n        invalid = self.store.get(state[\'session\'])\n        invalid.set_piece_at(chess.D2, invalid.remove_piece_at(chess.E2))\n        self.error(\'DESYNC\', lambda: self.store.sync(state[\'session\'], invalid.board_fen()))\n        self.assertEqual(self.store.get(state[\'session\']).fen(en_passant=\'fen\'), state[\'fen\'])\n\n    def test_history_retains_repetition(self):\n        state = self.store.create(chess.Board().board_fen(), \'w\')\n        for move in [\'g1f3\', \'g8f6\', \'f3g1\', \'f6g8\'] * 2:\n            state = self.advance(state, move)\n        board = self.store.get(state[\'session\'])\n        self.assertEqual(len(board.move_stack), 8)\n        self.assertTrue(board.is_repetition(3))\n\n    def test_stale_revision_and_unknown_session(self):\n        state = self.store.create(chess.Board().board_fen(), \'w\')\n        state = self.advance(state, \'e2e4\')\n        self.error(\'STALE\', lambda: self.store.sync(state[\'session\'], self.store.get(state[\'session\']).board_fen(), revision=state[\'revision\'] - 1))\n        self.error(\'SESSION_EXPIRED\', lambda: self.store.get(\'unknown\'))\n        self.error(\'SESSION_EXPIRED\', lambda: self.store.sync(\'unknown\', chess.Board().board_fen()))\n\n    def test_get_returns_isolated_copy_with_history(self):\n        state = self.store.create(chess.Board().board_fen(), \'w\')\n        state = self.advance(state, \'e2e4\')\n        copy = self.store.get(state[\'session\'])\n        self.assertEqual(len(copy.move_stack), 1)\n        copy.push_uci(\'e7e5\')\n        original = self.store.get(state[\'session\'])\n        self.assertEqual(original.fen(en_passant=\'fen\'), state[\'fen\'])\n        self.assertEqual(len(original.move_stack), 1)\n\n    def test_fools_mate_terminal_position(self):\n        state = self.store.create(chess.Board().board_fen(), \'w\')\n        for move in [\'f2f3\', \'e7e5\', \'g2g4\', \'d8h4\']:\n            state = self.advance(state, move)\n        self.assertTrue(self.store.get(state[\'session\']).is_checkmate())\n\n    def test_zero_ttl_expires_session(self):\n        store = PositionStore(ttl=0)\n        state = store.create(chess.Board().board_fen(), \'w\')\n        self.error(\'SESSION_EXPIRED\', lambda: store.get(state[\'session\']))\n\n    def test_session_limit_is_enforced(self):\n        store = PositionStore(max_sessions=1)\n        state = store.create(chess.Board().board_fen(), \'w\')\n        self.error(\'SESSION_LIMIT\', lambda: store.create(chess.Board().board_fen(), \'w\'))\n        self.assertEqual(store.get(state[\'session\']).fen(), chess.STARTING_FEN)\n\n    def test_initial_fen_turn_is_authoritative(self):\n        fen = \'7k/8/8/8/8/8/8/7K b - - 0 1\'\n        state = self.store.create(chess.Board(fen).board_fen(), \'w\', initial_fen=fen)\n        self.assertFalse(self.store.get(state[\'session\']).turn)\n\n    def test_initial_fen_must_match_and_be_valid(self):\n        placement = chess.Board().board_fen()\n        for fen in [\'8/8/8/8/8/8/8/8 w - - 0 1\', \'not a fen\']:\n            with self.subTest(fen=fen), self.assertRaises(PositionError):\n                self.store.create(placement, \'w\', initial_fen=fen)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', encoding='utf-8')
target = BASE / 'tests/test_server.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""HTTP contracts with fake UCI engine; no real engine operations."""\nimport http.client\nimport json\nimport sys\nimport threading\nimport unittest\nfrom pathlib import Path\nfrom unittest.mock import patch\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \'termux\'))\nimport chess\nimport chess.engine\nimport server\n\n\nclass FakeEngine:\n    id = {\'name\': \'ContractEngine\'}\n\n    def __init__(self, failed=False):\n        self.failed = failed\n        self.closed = False\n        self.calls = []\n\n    def configure(self, options):\n        self.options = options\n\n    def analyse(self, board, limit, multipv=1):\n        if self.failed:\n            raise chess.engine.EngineTerminatedError(\'terminated\')\n        self.calls.append((board.copy(), limit, multipv))\n        return [{\'score\': chess.engine.PovScore(chess.engine.Cp(30 - i), board.turn),\n                 \'pv\': [move], \'depth\': 5, \'nodes\': 100, \'time\': 0.01}\n                for i, move in enumerate(list(board.legal_moves)[:multipv])]\n\n    def close(self):\n        self.closed = True\n\n\nclass HTTPTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        cls.engine = FakeEngine()\n        cls.httpd = server.BoundedServer((\'127.0.0.1\', 0), server.Handler)\n        cls.httpd.service = server.Service(cls.engine, \'test-secret\')\n        cls.thread = threading.Thread(target=cls.httpd.serve_forever, kwargs={\'poll_interval\': 0.01}, daemon=True)\n        cls.thread.start()\n\n    @classmethod\n    def tearDownClass(cls):\n        cls.httpd.shutdown()\n        cls.httpd.server_close()\n        cls.thread.join(timeout=2)\n\n    def request(self, path, payload=None, auth=True, raw=None, method=\'POST\'):\n        conn = http.client.HTTPConnection(*self.httpd.server_address, timeout=3)\n        headers = {\'Content-Type\': \'application/json\'}\n        if auth:\n            headers[\'Authorization\'] = \'Bearer test-secret\'\n        body = raw if raw is not None else json.dumps(payload).encode() if payload is not None else None\n        conn.request(method, path, body=body, headers=headers)\n        response = conn.getresponse()\n        status, out_headers, data = response.status, dict(response.getheaders()), json.loads(response.read())\n        conn.close()\n        return status, out_headers, data\n\n    def new_session(self):\n        status, _, state = self.request(\'/session\', {\'placement\': chess.Board().board_fen(), \'turn\': \'w\'})\n        self.assertEqual(status, 200)\n        return state\n\n    def test_auth_and_health_v2(self):\n        self.assertEqual(self.request(\'/health\', auth=False, method=\'GET\')[0], 401)\n        status, headers, result = self.request(\'/health\', method=\'GET\')\n        self.assertEqual(status, 200)\n        self.assertEqual(result[\'version\'], \'2.0.0\')\n        self.assertTrue(result[\'sessions\'])\n        self.assertEqual(headers[\'Cache-Control\'], \'no-store\')\n\n    def test_session_sync_and_analysis_schema(self):\n        state = self.new_session()\n        board = chess.Board()\n        board.push_uci(\'e2e4\')\n        status, _, state = self.request(\'/sync\', {\'session\': state[\'session\'], \'revision\': state[\'revision\'], \'placement\': board.board_fen()})\n        self.assertEqual(status, 200)\n        self.assertEqual(state[\'revision\'], 1)\n        status, _, result = self.request(\'/analyse\', {\'session\': state[\'session\'], \'revision\': 1})\n        self.assertEqual(status, 200)\n        self.assertEqual(result[\'revision\'], 1)\n        self.assertIn(chess.Move.from_uci(result[\'move\']), board.legal_moves)\n        for key in [\'pv\', \'eval\', \'mate\', \'multipv\', \'style\', \'version\', \'requested_time\', \'search_budget\']:\n            self.assertIn(key, result)\n        self.assertEqual(result[\'eval\'], -0.3)\n\n    def test_invalid_requests(self):\n        cases = [{\'fen\': \'invalid\'}, {\'fen\': chess.STARTING_FEN, \'style\': \'invalid\'},\n                 {\'fen\': chess.STARTING_FEN, \'time\': float(\'nan\')},\n                 {\'fen\': chess.STARTING_FEN, \'time\': True},\n                 {\'fen\': chess.STARTING_FEN, \'aggression\': True},\n                 {\'fen\': chess.STARTING_FEN, \'aggression\': float(\'nan\')}, []]\n        for case in cases:\n            with self.subTest(case=case):\n                self.assertEqual(self.request(\'/analyse\', case)[0], 400)\n        self.assertEqual(self.request(\'/analyse\', raw=b\'{\')[0], 400)\n        self.assertEqual(self.request(\'/analyse\', raw=b\'x\' * 4097)[0], 413)\n        self.assertEqual(self.request(\'/analyse\', raw=b\'\')[0], 413)\n\n    def test_busy_retry_header(self):\n        self.httpd.service.lock.acquire()\n        try:\n            status, headers, result = self.request(\'/analyse\', {\'fen\': chess.STARTING_FEN})\n        finally:\n            self.httpd.service.lock.release()\n        self.assertEqual(status, 429)\n        self.assertEqual(headers[\'Retry-After\'], \'1\')\n        self.assertEqual(result[\'error\'], \'BUSY\')\n\n    def test_close_unknown_and_stale(self):\n        self.assertEqual(self.request(\'/close\', {\'session\': \'unknown\'})[0], 200)\n        state = self.new_session()\n        status, _, result = self.request(\'/analyse\', {\'session\': state[\'session\'], \'revision\': 99})\n        self.assertEqual(status, 409)\n        self.assertEqual(result[\'error\'], \'STALE\')\n        self.request(\'/close\', {\'session\': state[\'session\']})\n        status, _, result = self.request(\'/analyse\', {\'session\': state[\'session\'], \'revision\': 0})\n        self.assertEqual(status, 409)\n        self.assertEqual(result[\'error\'], \'SESSION_EXPIRED\')\n\n    def test_terminal_fools_mate(self):\n        state = self.new_session()\n        board = chess.Board()\n        for move in [\'f2f3\', \'e7e5\', \'g2g4\', \'d8h4\']:\n            board.push_uci(move)\n            status, _, state = self.request(\'/sync\', {\'session\': state[\'session\'], \'revision\': state[\'revision\'], \'placement\': board.board_fen()})\n            self.assertEqual(status, 200)\n        status, _, result = self.request(\'/analyse\', {\'session\': state[\'session\'], \'revision\': state[\'revision\']})\n        self.assertEqual(status, 200)\n        self.assertTrue(result[\'game_over\'])\n        self.assertEqual(result[\'result\'], \'0-1\')\n        self.assertEqual(result[\'reason\'], \'CHECKMATE\')\n\n    def test_style_and_force_precise(self):\n        service = server.Service(FakeEngine(), \'key\')\n        result = service.analyse({\'fen\': chess.STARTING_FEN, \'style\': \'attacking\', \'aggression\': 50})\n        self.assertEqual(result[\'multipv\'], 4)\n        self.assertEqual(result[\'style\'], \'attacking\')\n        self.assertLessEqual(result[\'evaluation_loss_cp\'], 25)\n        precise = server.Service(FakeEngine(), \'key\', force_precise=True)\n        result = precise.analyse({\'fen\': chess.STARTING_FEN, \'style\': \'attacking\', \'aggression\': 50})\n        self.assertEqual(result[\'multipv\'], 1)\n        self.assertEqual(result[\'style\'], \'precise\')\n        self.assertEqual(result[\'aggression\'], 0)\n        self.assertFalse(result[\'personality_supported\'])\n\n\nclass EngineRecoveryTests(unittest.TestCase):\n    def test_terminated_process_is_replaced_once(self):\n        broken, healthy = FakeEngine(failed=True), FakeEngine()\n        with patch(\'server.chess.engine.SimpleEngine.popen_uci\', side_effect=[broken, healthy]) as spawn:\n            manager = server.EngineManager(\'stockfish\', 2, 128)\n            result = manager.analyse(chess.Board(), chess.engine.Limit(time=0.1), multipv=1)\n            self.assertTrue(result[0][\'pv\'])\n            self.assertTrue(broken.closed)\n            self.assertEqual(spawn.call_count, 2)\n            manager.close()\n            self.assertTrue(healthy.closed)\n\n    def test_second_failure_propagates(self):\n        first, second = FakeEngine(failed=True), FakeEngine(failed=True)\n        with patch(\'server.chess.engine.SimpleEngine.popen_uci\', side_effect=[first, second]) as spawn:\n            manager = server.EngineManager(\'stockfish\', 2, 128)\n            with self.assertRaises(chess.engine.EngineTerminatedError):\n                manager.analyse(chess.Board(), chess.engine.Limit(time=0.1))\n            self.assertEqual(spawn.call_count, 2)\n            self.assertTrue(first.closed)\n            manager.close()\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', encoding='utf-8')
target = BASE / 'tests/test_castling_animation.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text("# Python / test_castling_animation.py / Python 3.10+ / late castling animation regression.\nimport sys\nimport unittest\nfrom pathlib import Path\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'termux'))\nimport chess\nfrom positions import PositionStore\n\nclass CastlingAnimationTests(unittest.TestCase):\n    def test_rook_arriving_before_king_is_corrected_atomically(self):\n        for turn, rook_move, castle in [('w','h1f1','e1g1'),('w','a1d1','e1c1'),\n                                       ('b','h8f8','e8g8'),('b','a8d8','e8c8')]:\n            with self.subTest(castle=castle):\n                before=chess.Board(f'r3k2r/8/8/8/8/8/8/R3K2R {turn} KQkq - 0 1')\n                store=PositionStore()\n                data=store.create(before.board_fen(),turn,before.fen())\n                transient=before.copy(); transient.push_uci(rook_move)\n                first=store.sync(data['session'],transient.board_fen(),0)\n                final=before.copy(); final.push_uci(castle)\n                result=store.sync(data['session'],final.board_fen(),first['revision'])\n                actual=store.get(data['session'])\n                self.assertEqual(actual.fen(en_passant='fen'),final.fen(en_passant='fen'))\n                self.assertEqual([m.uci() for m in actual.move_stack],[castle])\n                self.assertEqual(result['last_move'],castle)\n\nif __name__=='__main__': unittest.main()\n", encoding='utf-8')
SERVER = BASE / 'termux' / 'server.py'
subprocess.run([sys.executable, str(BASE / 'termux' / 'install_engine.py')], check=True)
ENGINE = BASE / 'termux' / 'bin' / 'stockfish'
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(BASE/'tests')], check=True)
subprocess.run([sys.executable, str(BASE/'termux'/'check_engine.py'), '--engine', str(ENGINE)], check=True)
TUNNEL = BASE / 'cloudflared'
EXPECTED = 'd33ff2d14475178d2012c2c56beba87389ac5ded27649519f198a7d3134a99db'
if not TUNNEL.exists() or hashlib.sha256(TUNNEL.read_bytes()).hexdigest() != EXPECTED:
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/download/2026.10.0/cloudflared-linux-amd64', TUNNEL)
if hashlib.sha256(TUNNEL.read_bytes()).hexdigest() != EXPECTED:
    raise RuntimeError('SHA256 do Cloudflare Tunnel inválido')
TUNNEL.chmod(0o755)
print('Instalação e testes concluídos. Execute a célula de início.')


In [ ]:
# Iniciar: esta célula permanece executando enquanto o servidor estiver ligado.
#@title Configuração e início
THREADS = min(2, os.cpu_count() or 1)
HASH_MB = 128
import time, json, secrets, threading, queue, re, socket
with socket.socket() as sock:
    sock.bind(("127.0.0.1", 0))
    PORT = sock.getsockname()[1]
LOCAL_URL = f"http://127.0.0.1:{PORT}"
if not 1 <= THREADS <= min(16, os.cpu_count() or 1):
    raise ValueError("Threads devem estar entre 1 e o total disponível: " + str(os.cpu_count()))
if not 16 <= HASH_MB <= 65536:
    raise ValueError("Hash deve estar entre 16 e 65536 MB")
key_file = BASE / "termux" / ".api-key"
if not key_file.exists():
    key_file.write_text(secrets.token_urlsafe(32))
key_file.chmod(0o600)
key = key_file.read_text().strip()
server = tunnel = None
output = queue.Queue()
def collect(process, name):
    for line in process.stdout:
        output.put((name, line.rstrip()))
try:
    server = subprocess.Popen(
        [sys.executable, "-u", str(SERVER), "--engine", str(ENGINE),
         "--port", str(PORT), "--threads", str(THREADS), "--hash", str(HASH_MB)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    threading.Thread(target=collect, args=(server, "motor"), daemon=True).start()
    for attempt in range(60):
        if server.poll() is not None:
            raise RuntimeError("Servidor encerrou; confira as mensagens abaixo")
        try:
            req = urllib.request.Request(LOCAL_URL + "/health",
                                         headers={"Authorization": "Bearer " + key})
            with urllib.request.urlopen(req, timeout=2) as response:
                health = json.load(response)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("Servidor não respondeu em 60 segundos")
    print("Motor:", health["engine"].get("name"))
    print("Threads:", THREADS, "| Hash:", HASH_MB, "MB")
    print("CHAVE PARA O PAINEL:", key)
    tunnel = subprocess.Popen(
        [str(TUNNEL), "tunnel", "--protocol", "http2", "--url", LOCAL_URL],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    threading.Thread(target=collect, args=(tunnel, "túnel"), daemon=True).start()
    print("Aguardando URL HTTPS...")
    while server.poll() is None and tunnel.poll() is None:
        try:
            name, line = output.get(timeout=1)
        except queue.Empty:
            continue
        if "Chave para colar" in line:
            continue
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if match:
            print("\nURL PARA O PAINEL:", match.group(0), "\n", flush=True)
        elif name == "motor" or "ERR" in line or "Registered tunnel" in line:
            print(line, flush=True)
    raise RuntimeError("Motor ou túnel encerrou. Execute esta célula novamente.")
except KeyboardInterrupt:
    print("Parando servidor e túnel.")
finally:
    while not output.empty():
        name, line = output.get_nowait()
        if "Chave para colar" not in line:
            print(line)
    for process in (tunnel, server):
        if process and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
